In [238]:
import pandas as pd
import numpy as np

In [239]:
# afficher les résultats en entier
pd.set_option('display.max_row',100)
pd.set_option('display.max_column', 50)

## Nettoyage des données et analyse descriptive

In [240]:
# Chargement du fichier csv
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [241]:
# Visualisation des variables, des valeurs nulles et des types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [242]:
# supprimer les lignes avec la variable "TotalCharges" nulle
df.dropna(subset=["TotalCharges"], inplace=True)

In [243]:
# détection des valeurs manquantes
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [244]:
# Visualisation des valeurs de chaque variable
for field in df:
    print(f"{field}:")
    print(df[f"{field}"].unique())

customerID:
['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
gender:
['Female' 'Male']
SeniorCitizen:
[0 1]
Partner:
['Yes' 'No']
Dependents:
['No' 'Yes']
tenure:
[ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26 39]
PhoneService:
['No' 'Yes']
MultipleLines:
['No phone service' 'No' 'Yes']
InternetService:
['DSL' 'Fiber optic' 'No']
OnlineSecurity:
['No' 'Yes' 'No internet service']
OnlineBackup:
['Yes' 'No' 'No internet service']
DeviceProtection:
['No' 'Yes' 'No internet service']
TechSupport:
['No' 'Yes' 'No internet service']
StreamingTV:
['No' 'Yes' 'No internet service']
StreamingMovies:
['No' 'Yes' 'No internet service']
Contract:
['Month-to-month' 'One year' 'Two year']
PaperlessBilling:
['Yes' 'No']
PaymentMethod:
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 '

In [245]:
# visualisation de la variable target
df['Churn'].value_counts()

Churn
No     5163
Yes    1869
Name: count, dtype: int64

In [246]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

### Synthèse de l'analyse descriptive
- le dataset comprend 7032 lignes, (après la suppression de 11 lignes ayant une valeur nulle à la variable 'TotalCharges')
- le dataset comprend 20 variables (dont 16 de type 'object', 2 de type 'float64' et 2 de type 'int64')
- il y a un déséquilibre de classe sur la variable 'Churn' (target) (avec 5163 de 'No' et 1869 de 'Yes')

## Pré-traitement des données

In [247]:
# import des librairies pour le feature engineering
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

In [248]:
# Encodage de la variable 'Churn' (future target), avec 'No' = 0 et 'Yes'= 1, conversion de type en 'int'
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [249]:
# Suppression de la colonne "customerID" car non nécessaire
df.drop(columns=["customerID"], inplace=True)

In [250]:
# Identification des variables numériques et catégorielles de la variable X pour le pré-traitement
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns
categorical_features = df.select_dtypes(include=['object']).columns

# suppression de la variable "Churn" de numerical_features car ce n'est pas une feature
numerical_features = numerical_features[:3]

In [251]:
print(categorical_features)
print()
print(numerical_features)

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')


In [252]:
# Encodage des colonnes catégorielles avec LabelEncoder
for col in categorical_features:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col])

In [253]:
# Standardisation des colonnes numériques avec StandardScaler
standard_scaler = StandardScaler()
df[numerical_features] = standard_scaler.fit_transform(df[numerical_features])

In [254]:
df.sample(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
1426,1,2.271039,1,0,1.571829,1,2,1,2,2,2,2,0,0,1,1,2,0.904200,6585.20,0
6103,1,-0.440327,0,0,-0.669089,1,0,2,1,1,1,1,1,1,0,0,1,-1.497422,294.90,0
4371,1,-0.440327,1,1,-0.791321,1,2,2,1,1,1,1,1,1,1,0,0,-1.339530,343.60,0
660,0,-0.440327,0,0,-1.158016,0,1,0,0,0,0,0,0,0,0,1,3,-1.347840,96.05,1
4209,1,-0.440327,1,1,0.471742,1,0,0,0,2,0,2,0,0,0,1,0,-0.329021,2549.10,0


## Implémentation du modèle de deep learning

In [255]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split

In [256]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [257]:
# Séparation des features (X) de la target (y)

# Filtrage des colonnes features
features = df.iloc[:, 0:-1]
# conversion de la variable features en array numpy (X) pour la manipulation avec Pytorch
X = features.to_numpy()

# Filtrage de la colonne target
target = df.iloc[:, -1]
# conversion de la variable target en array numpy (y) pour la manipulation avec Pytorch
y = target.to_numpy()

In [258]:
# Visualisation des variables créées
print("features : ", features.shape)
print("X : ", X.shape)
print("target : ", target.shape)
print("y : ", y.shape)

features :  (7032, 19)
X :  (7032, 19)
target :  (7032,)
y :  (7032,)


In [259]:
# Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [260]:
# Vérification
print("X_train : ", X_train.shape)
print("y_train : ", y_train.shape)
print("X_test : ", X_test.shape)
print("y_test : ", y_test.shape)

X_train :  (5625, 19)
y_train :  (5625,)
X_test :  (1407, 19)
y_test :  (1407,)


In [261]:
# Conversion des variables X_train et y_train en Tensor (permettant la manipulation avec PyTorch)
dataset_train = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))

# Conversion des variables X_test et y_test en Tensor (permettant la manipulation avec PyTorch)
dataset_test = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

In [262]:
print("dataset_train : ", dataset_train)
print("dataset_test : ", dataset_test)

dataset_train :  <torch.utils.data.dataset.TensorDataset object at 0x000001CB3CD55130>
dataset_test :  <torch.utils.data.dataset.TensorDataset object at 0x000001CB3CCE6810>


In [263]:
# nombre d'échantillons inclus à chaque itération
batch_size = 32

# randomisation des données lors des itérations
shuffle = True

# Chargement des DataLoaders d'entrainement et de test
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=shuffle)
dataloader_test = DataLoader(dataset_test, batch_size=batch_size)

In [264]:
# Gestion du déséquilibre de classe
n_pos = (df['Churn'] == 1).sum()
n_neg = (df['Churn'] == 0).sum()
pos_weight_value = n_neg / n_pos
pos_weight = torch.tensor([pos_weight_value]).to(device)

In [265]:
# Conception du modèle de réseaux de neurones
class ChurnModel(nn.Module):
    def __init__(self, input_dim):
        super(ChurnModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)  # PAS de Sigmoid ici
        )


    def forward(self, x):
        return self.model(x)  # logits


In [266]:
# Initialisation modèle / loss / optimiser
model = ChurnModel(input_dim=X.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [267]:
# Boucle d'entraînement
n_epochs = 100
for epoch in range(n_epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in dataloader_train:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).float()

        optimizer.zero_grad()
        logits = model(X_batch).view(-1)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{n_epochs}, Loss: {total_loss:.4f}")

Epoch 1/100, Loss: 1194.2712
Epoch 2/100, Loss: 211.2226
Epoch 3/100, Loss: 194.9399
Epoch 4/100, Loss: 185.2737
Epoch 5/100, Loss: 180.5057
Epoch 6/100, Loss: 183.5094
Epoch 7/100, Loss: 183.4241
Epoch 8/100, Loss: 179.9163
Epoch 9/100, Loss: 180.4574
Epoch 10/100, Loss: 178.4485
Epoch 11/100, Loss: 178.5817
Epoch 12/100, Loss: 177.6125
Epoch 13/100, Loss: 178.0327
Epoch 14/100, Loss: 177.2232
Epoch 15/100, Loss: 178.9382
Epoch 16/100, Loss: 178.8976
Epoch 17/100, Loss: 179.4332
Epoch 18/100, Loss: 178.2182
Epoch 19/100, Loss: 177.1641
Epoch 20/100, Loss: 178.1717
Epoch 21/100, Loss: 178.4905
Epoch 22/100, Loss: 178.5917
Epoch 23/100, Loss: 178.6998
Epoch 24/100, Loss: 178.0257
Epoch 25/100, Loss: 176.2640
Epoch 26/100, Loss: 176.9930
Epoch 27/100, Loss: 176.9358
Epoch 28/100, Loss: 176.2376
Epoch 29/100, Loss: 175.8167
Epoch 30/100, Loss: 175.3682
Epoch 31/100, Loss: 174.1778
Epoch 32/100, Loss: 175.0888
Epoch 33/100, Loss: 173.0617
Epoch 34/100, Loss: 173.8737
Epoch 35/100, Loss: 17

## Evaluation du modèle

In [268]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, roc_auc_score, recall_score

In [269]:
# Evaluation du modèle
model.eval()

y_true = []
y_probs = []

with torch.no_grad():
    for X_batch, y_batch in dataloader_test:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).float()

        logits = model(X_batch).view(-1)
        probs = torch.sigmoid(logits)

        y_true.extend(y_batch.cpu().numpy())
        y_probs.extend(probs.cpu().numpy())


# Conversion
y_true = np.array(y_true)
y_pred = (np.array(y_probs) > 0.5).astype(int)


In [270]:
# Visualisation

# Métriques
f1 = f1_score(y_true, y_pred)
roc = roc_auc_score(y_true, y_probs)
recall = recall_score(y_true, y_pred)

print(f"F1-score      : {f1:.4f}")
print(f"ROC-AUC       : {roc:.4f}")
print(f"Recall (churn): {recall:.4f}")

F1-score      : 0.6161
ROC-AUC       : 0.8273
Recall (churn): 0.7273


In [271]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

         0.0       0.89      0.77      0.82      1033
         1.0       0.53      0.73      0.62       374

    accuracy                           0.76      1407
   macro avg       0.71      0.75      0.72      1407
weighted avg       0.79      0.76      0.77      1407

